### Data Load

In [ ]:
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

- 0. 어떤 데이터가 들어있는지 탐색하세요.
- 0. Description Featrure 는 삭제하세요.

In [ ]:
train_df.head()

In [ ]:
del_Desc = train_df.drop('Description', axis=1, inplace=False)
del_Desc

### Data EDA

- 1. Print Data shape 
- 2. 데이터 정보 infor 를 보고 데이터를 파악하세요.

결측치 유무, 수치형/범주형 데이터 파악하고 markdown을 이용해서아래에 결과 summary 하세요.

In [ ]:
print(del_Desc.shape)
del_Desc.info()
del_Desc.isna().sum()

##### 결측치 - 없음
##### 수치형 데이터 - Age, Fee, PhotoAmt, AdoptionSpeed
##### 범주형 데이터 - Breed1, Gender, Color1, Color2, MaturitySize, FurLength, Vaccinated, Sterilized, Health

- 3. 수치형 변수 | 범주형 변수를 나누세요. 

In [ ]:
TARGET = "AdoptionSpeed"
feature_cols = [c for c in train_df.columns if c != TARGET]

num_cols = train_df[feature_cols].select_dtypes(include=["number"]).columns.tolist()
obj_cols = train_df[feature_cols].select_dtypes(exclude=["number"]).columns.tolist()

print(num_cols)
print(obj_cols)


- 4. 수치형 변수를 시각화하고 각 데이터에 대해 전처리 방법을 고안해보세요. (적용할 것이 없으면 넘어감)

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(4 * len(num_cols), 4))

for ax, col in zip(axes, num_cols):
    sns.histplot(train_df[col], ax = ax)
plt.tight_layout()
plt.show()
vc_Age = train_df['Age'].value_counts().sort_index()
vc_Fee = train_df['Fee'].value_counts().sort_index()
vc_PhotoAmt = train_df['PhotoAmt'].value_counts().sort_index()

vc_Age, vc_Fee, vc_PhotoAmt

- 5. 클래스 분포 시각화하고 데이터 분석에 있어서 우려되는 점을 기술하세요.

클래스 불균형: Target인 AdoptionSpeed 구간이 적으면 macro-F1 등에서 불리할 ㅅㅜ 있기 때문에 class_weight나 샘플 가중으로 완화

In [ ]:
vc = train_df[TARGET].value_counts().sort_index()
sns.barplot(x=vc.index.astype(str), y=vc.values)
plt.title("AdoptionSpeed 분포")
plt.xlabel(TARGET)
plt.ylabel("count")
plt.show()
print(vc)

### v2 연결 (과제 6번 이전)

- `problem.ipynb` 구간(5번까지) 이후 `v1.ipynb` 코드를 이어 붙였습니다.
- `Description`은 이후 파이프라인용으로 제거하고, `v1`과 동일하게 `numeric_cols` / `categorical_cols`를 정의합니다.
- `test.csv`는 상단 `Data Load`에서 읽어도 되고, **추론·제출 직전**에만 `pd.read_csv("test.csv")`로 읽어도 됩니다(동일 경로·동일 컬럼이면 결과 동일).


In [ ]:
import numpy as _np
RANDOM_STATE = 42
train_df = train_df.drop(columns=["Description"], errors="ignore")
test_df = test_df.drop(columns=["Description"], errors="ignore")
feature_cols = [c for c in train_df.columns if c != TARGET]
numeric_cols = train_df[feature_cols].select_dtypes(include=[_np.number]).columns.tolist()
categorical_cols = train_df[feature_cols].select_dtypes(exclude=[_np.number]).columns.tolist()
print("numeric_cols:", numeric_cols)
print("categorical_cols:", categorical_cols)


- 6. 범주형 변수 분포를 시각화하고 전처리 방법은 고안해보세요. (적용할 것이 없으면 넘어감)
- 7. 범주형 변수처리에 있어 Label encoding, onehot encoding 방식이 있다. 아래 내용을 고민해보세요.

    - Label encoding을 적용할 경우 어울리는 모델은?
    - Label encoding을 적용할 경우 우려되는 점은?
    - Onehot encoding을 해야되는 모델은?
    - Breed1 는 어떻게 처리하면 좋을지 고민하세요.

--------------------------------------------------
- 6. Visualize the distribution of categorical variables and consider appropriate preprocessing methods. If no preprocessing is needed, you may skip it.
- 7. There are two common methods for handling categorical variables: Label Encoding and One-Hot Encoding. Consider the following questions:
    - Which models are suitable when Label Encoding is applied?
    - What concerns may arise when Label Encoding is applied?
    - Which models require One-Hot Encoding?
    - Consider how Breed1 should be handled.


**인코딩 메모**

- **트리 기반(RF, HGB)**: 순서가 없는 범주에 Label encoding을 쓰면 가짜 순서관계가 생길 수 있어 One-Hot 또는 고카디널리티는 빈도/상위K+Other가 안전.
- **로지스틱/선형**: 범주는 One-Hot(희소)이 일반적.
- **Breed1**: 수준이 많으므로 `OneHotEncoder(max_categories=..., handle_unknown='ignore')`로 상위 빈도만 유지하고 나머지 묶음 처리.


In [ ]:
n_cat = len(categorical_cols)
ncols = 3
nrows = int(np.ceil(n_cat / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(categorical_cols):
    ax = axes[i]
    top = train_df[col].value_counts().head(15)
    sns.barplot(x=top.values, y=top.index.astype(str), ax=ax)
    ax.set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


- 8. 위 인코딩 전략을 적용하여 실제로 인코딩을 진행하세요.
--------------------------------
- 8. Apply the encoding strategy above and actually perform the encoding.


In [ ]:
# 인코딩은 아래 sklearn ColumnTransformer + Pipeline에서 일괄 적용합니다.
# (학습 데이터에만 fit, test는 동일 pipeline으로 transform)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    max_categories=25,
)
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


### Feature Selection


- 9. Correlation 분석을 실시하고 feature selection 진행하세요.
- 10. Mutual Information 이 무엇인지 파악하세요. Correlation과 차이점을 파악하세요.
- 11. 9번에 이어서 Mutual Information 기반의 feature selection 진행하세요. (시각화도 하세요.)
-----------------------------------
- 9. Conduct correlation analysis and perform feature selection.
- 10. Understand what Mutual Information is and identify the differences between Mutual Information and Correlation.
- 11. Following Question 9, perform feature selection based on Mutual Information. Also visualize the results.


**상관 vs Mutual Information**

- **Correlation(Pearson)**: 선형 관계 강도. 범주형에는 부적합할 수 있음.
- **Mutual Information**: 타깃과의 (선형/비선형 포함) 통계적 의존도를 정보이론적으로 측정. 범주·비선형에 유리.


In [ ]:
X_num = train_df[numeric_cols]
corr = X_num.corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("수치형 변수 상관행렬")
plt.show()

# 수치형만으로는 제한적이므로, 본 노트북은 전체 피처를 유지하고 MI로 중요도만 참고합니다.
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif

X_mi = train_df[feature_cols].copy()
for c in categorical_cols:
    X_mi[c] = X_mi[c].astype(str)
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_mi[categorical_cols] = ord_enc.fit_transform(X_mi[categorical_cols])
mi = mutual_info_classif(X_mi, train_df[TARGET], random_state=RANDOM_STATE)
mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=mi_series.values, y=mi_series.index)
plt.title("Mutual Information (Ordinal 임시 인코딩 기준)")
plt.tight_layout()
plt.show()
print(mi_series)


### Modeling


- 12. 위 학습 전략을 적용하여 전체 데이터 처리 파이프라인 만들고 데이터 정제하기
- 13. 학습을 위해 Train test split 을 이용해 데이터 8:2로 나누기
--------------------------------
- 12. Build a complete data preprocessing pipeline based on the strategies above and clean the dataset.
- 13. Split the data into training and test sets using an 8:2 ratio with train_test_split.


In [ ]:
from sklearn.model_selection import train_test_split

X = train_df[feature_cols]
y = train_df[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_valid.shape)


- 14. 적절한 모델 3가지 이상을 설정하고 학습하기 (모델 선택 이유는?)
--------------------------------
- 14. Select and train at least three appropriate models. Explain why you selected those models.


**모델 선택 이유**

- **RandomForest**: 범주+수치 혼합에 강하고 해석/튜닝이 쉬움.
- **HistGradientBoosting**: 부스팅으로 비선형 패턴 포착, 대용량에 효율적.
- **LogisticRegression(multinomial)**: 선형 베이스라인으로 과적합 위험이 상대적으로 낮고 비교 기준으로 유용.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_depth=8,
        learning_rate=0.08,
        max_iter=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=RANDOM_STATE,
    ),
}

fitted = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_valid)
    print(f"=== {name} ===")
    print("accuracy:", accuracy_score(y_valid, pred))
    print("macro F1:", f1_score(y_valid, pred, average="macro"))
    print(classification_report(y_valid, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_valid, pred)
    plt.title(name)
    plt.show()


### Evaluation


- 15. 모델 평가 메트릭을 정하고 평가하기, 평가결과 분석하기 (시각화 필수)
- 16. Class weight / Optuna 적용해서 최적화하기
--------------------------------
- 15. Choose suitable model evaluation metrics, evaluate the models, and analyze the evaluation results. Visualization is required.
- 16. Apply class_weight and/or Optuna to optimize the model.

리더보드에서 공개하는 평가지표(예: QWK 등)가 다르면 해당 지표에 맞게 손실/후처리를 조정하면 됩니다.


In [ ]:
# 이미 class_weight='balanced' 적용됨. Optuna로 RandomForest 하이퍼파라미터 탐색 예시.
try:
    import optuna
except ImportError:
    optuna = None
    print("optuna 미설치: pip install optuna 후 재실행 가능")

def run_optuna_rf(n_trials=15):
    def objective(trial):
        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 400),
            max_depth=trial.suggest_int("max_depth", 8, 40),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_valid)
        return f1_score(y_valid, pred, average="macro")

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study

if optuna is not None:
    study = run_optuna_rf(n_trials=12)
    print("best macro-F1:", study.best_value)
    print("best params:", study.best_params)
    best_rf = RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
        **study.best_params,
    )
    best_pipe = Pipeline([("prep", preprocessor), ("clf", best_rf)])
    best_pipe.fit(X_train, y_train)
    best_name = "RandomForest_Optuna"
    fitted[best_name] = best_pipe
    pred = best_pipe.predict(X_valid)
    print("Optuna RF valid macro F1:", f1_score(y_valid, pred, average="macro"))
else:
    best_name = None


**최종 모델 선택**: 검증 macro-F1이 가장 높은 파이프라인을 아래에서 자동 선택합니다.


In [ ]:
scores = {}
for name, pipe in fitted.items():
    pred = pipe.predict(X_valid)
    scores[name] = f1_score(y_valid, pred, average="macro")
best_model_name = max(scores, key=scores.get)
best_model = fitted[best_model_name]
print("모델별 valid macro-F1:", scores)
print("선택:", best_model_name)


### Interpretation with SHAP


- 17. shap을 이용해 결과 분석하기
    - 입양 속도에 영향을 미치는 중요한 인자는?
    - 해당인자가 어떤 값을 가질 때 입양속도가 빨라지는가?
    - 통제 가능한 변수는 무엇이고 어떻게 활용 하면 좋은가?
--------------------------------
- 17. Analyze the results using SHAP.
    - What are the important factors that affect adoption speed?
    - When those factors have certain values, does the adoption speed become faster?
    - What are the controllable variables, and how can they be effectively utilized?


In [ ]:
# 트리 모델이면 TreeExplainer 권장. 로지스틱이 선택된 경우 LinearExplainer 등으로 대체.
try:
    import shap
except ImportError:
    shap = None
    print("shap 미설치: pip install shap")

if shap is not None and best_model_name.startswith(("RandomForest", "HistGradient")):
    explainer = shap.TreeExplainer(best_model.named_steps["clf"])
    prep = best_model.named_steps["prep"]
    Xv_prep = prep.transform(X_valid.iloc[:500])
    feat_names = prep.get_feature_names_out()
    sv = explainer.shap_values(Xv_prep)
    # 다분류: 클래스별 리스트 → 샘플·클래스 평균 절대값으로 요약
    if isinstance(sv, list):
        mean_abs = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
    else:
        mean_abs = np.abs(sv).mean(axis=0)
    imp = pd.Series(mean_abs, index=feat_names).sort_values(ascending=False).head(20)
    plt.figure(figsize=(8, 6))
    sns.barplot(x=imp.values, y=imp.index)
    plt.title("SHAP mean(|value|) 상위 피처")
    plt.tight_layout()
    plt.show()
elif shap is not None:
    print("선택 모델이 트리가 아니면 SHAP LinearExplainer 등으로 바꿔 실행하세요.")


### Test 추론 및 제출 파일 (PDF: 컬럼명 `prediction`)


In [ ]:
# 전체 train으로 best 구조 재학습 후 test 예측
from sklearn.base import clone

final_model = clone(best_model)
final_model.fit(X, y)

X_test = test_df[feature_cols]
test_pred = final_model.predict(X_test)
submission = pd.DataFrame({"prediction": test_pred})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("saved: submission.csv")


- `submission.csv`는 리더보드 안내에 따라 업로드합니다.
- `test.csv`에는 정답이 없으므로, 내부 검증은 `train_test_split` 구간에서만 수행했습니다.
